# 01 · 보간법 기초 — 딥러닝 없이 어디까지 되나

선명한 위성사진(HR)을 **일부러 흐리게 줄인 뒤 다시 키워서**, 원본과 얼마나 닮았는지 잰다.
딥러닝을 쓰지 않고 고전적인 보간법 4가지만 쓴다.

| 방법 | 한 줄 설명 |
|---|---|
| **Nearest** | 가장 가까운 화소를 그대로 복사 |
| **Bilinear** | 주변 2×2 화소를 직선으로 섞음 |
| **Bicubic** | 주변 4×4 화소를 3차 곡선으로 섞음 |
| **Lanczos** | 주변 8×8 화소를 sinc 함수로 섞음 |

## 왜 이걸 먼저 하나

딥러닝 SR 이 정말 쓸모 있는지 판단하려면 **비교 대상**이 있어야 한다.
"PSNR 19 dB 나왔다"는 말은 그 자체로 아무 의미가 없고, "그냥 확대하면 18.2 dB 인데
19 dB 가 나왔다"여야 의미가 생긴다. 이 노트북에서 그 기준선을 만든다.

## 이 노트북에서 보는 것

1. 배율(×2, ×3, ×4)과 커널(4종)별 **PSNR·SSIM 표**
2. 확대해서 눈으로 확인 — **계단 현상 / 흐림 / 링잉**이 각각 어디서 나오는지
3. 경계 단면을 그래프로 그려 **링잉(overshoot)을 수치로** 확인
4. 딥러닝 SR 과 비교할 **baseline 수치 저장**

구글 드라이브도 `git clone` 도 `pip install` 도 쓰지 않는다. GPU 도 필요 없다.

## 1. 사진 받기

`SR_practice` 저장소의 **검증용 HR 사진 10장**을 쓴다. 딥러닝 SR 노트북이 점수를 매길 때
쓰는 것과 같은 사진이라, 여기서 나온 수치를 그대로 비교 대상으로 삼을 수 있다.

In [ ]:
import json, os, urllib.request
import numpy as np

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
API  = 'https://api.github.com/repos/BWMIN-Hub/SR_practice/contents/dataset/validation/HR'

with urllib.request.urlopen(API) as r:
    names = sorted(x['name'] for x in json.load(r))

os.makedirs('hr', exist_ok=True)
for n in names:
    if not os.path.exists(f'hr/{n}'):
        urllib.request.urlretrieve(f'{BASE}/dataset/validation/HR/{n}', f'hr/{n}')

import imageio.v2 as imageio
HR = {n[:-4]: imageio.imread(f'hr/{n}') for n in names}
h, w, _ = next(iter(HR.values())).shape
print(f'HR 사진 {len(HR)}장, 크기 {w}x{h}, 화소 크기 3.33 m')
print(f'{w} 는 2, 3, 4 로 모두 나누어떨어진다 -> 배율 3가지를 깔끔하게 실험할 수 있다')

## 2. 줄였다 키우기

**줄일 때**는 `INTER_AREA`(면적 평균)를 쓴다. 여러 화소를 평균내는 방식이라 실제 위성이
낮은 해상도로 찍는 것과 가장 비슷하다. 여기서 다른 방식을 쓰면 실험 자체가 왜곡된다.

**키울 때** 4가지 커널을 비교한다. 원래 크기로 되돌린 뒤 원본과 대조하면 되므로
정답이 항상 있다.

In [ ]:
import cv2

KERNELS = {
    'Nearest' : cv2.INTER_NEAREST,
    'Bilinear': cv2.INTER_LINEAR,
    'Bicubic' : cv2.INTER_CUBIC,
    'Lanczos' : cv2.INTER_LANCZOS4,
}
SCALES = [2, 3, 4]
SHAVE  = 4          # 가장자리는 보간이 불안정하므로 잘라내고 잰다

def downsample(hr, s):
    """HR -> LR. 면적 평균으로 줄인다."""
    H, W = hr.shape[:2]
    return cv2.resize(hr, (W // s, H // s), interpolation=cv2.INTER_AREA)

def upsample(lr, s, kernel):
    H, W = lr.shape[:2]
    return cv2.resize(lr, (W * s, H * s), interpolation=KERNELS[kernel])

# 한 장으로 시연
stem = list(HR)[0]
hr = HR[stem]
for s in SCALES:
    lr = downsample(hr, s)
    up = upsample(lr, s, 'Bicubic')
    print(f'x{s}:  HR {hr.shape[1]}x{hr.shape[0]}  ->  LR {lr.shape[1]}x{lr.shape[0]}  ->  복원 {up.shape[1]}x{up.shape[0]}')

## 3. 정량 비교 — PSNR과 SSIM

- **PSNR** — 화소 값이 얼마나 정확한가. dB 단위이고 높을수록 좋다. +1 dB 면 꽤 큰 차이다
- **SSIM** — 구조가 얼마나 닮았는가. 0~1 이고 1 이면 완전히 같다

PSNR 은 화소 하나하나의 오차만 보기 때문에 **흐릿한 결과에 후한 점수**를 준다.
사람 눈이 느끼는 선명함과 어긋나는 지점이라, SSIM 을 같이 본다.

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

def score(pred, gt):
    a, b = pred[SHAVE:-SHAVE, SHAVE:-SHAVE], gt[SHAVE:-SHAVE, SHAVE:-SHAVE]
    return psnr(b, a, data_range=255), ssim(b, a, data_range=255, channel_axis=2)

res = {}                        # res[(scale, kernel)] = (psnr, ssim)
per_scene = {}                  # per_scene[(scale, kernel)] = [psnr per scene]
for s in SCALES:
    for k in KERNELS:
        ps, ss = [], []
        for stem, hr in HR.items():
            up = upsample(downsample(hr, s), s, k)
            p, q = score(up, hr)
            ps.append(p); ss.append(q)
        res[(s, k)] = (float(np.mean(ps)), float(np.mean(ss)))
        per_scene[(s, k)] = ps

print(f'{"":10s}' + ''.join(f'{k:>22s}' for k in KERNELS))
print(f'{"":10s}' + ''.join(f'{"PSNR":>11s}{"SSIM":>11s}' for _ in KERNELS))
for s in SCALES:
    row = f'x{s:<9d}'
    for k in KERNELS:
        p, q = res[(s, k)]
        row += f'{p:11.2f}{q:11.4f}'
    print(row)

print('\n배율이 커질수록 점수가 떨어진다. 버려진 정보가 그만큼 많아서다.')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
x = np.arange(len(SCALES)); w = 0.2
colors = ['#c96a5b', '#d9a441', '#2f6f9f', '#4f9d69']
for j, (name, unit) in enumerate([('PSNR', 'dB'), ('SSIM', '')]):
    for i, k in enumerate(KERNELS):
        v = [res[(s, k)][j] for s in SCALES]
        ax[j].bar(x + (i - 1.5) * w, v, w, label=k, color=colors[i])
    ax[j].set_xticks(x); ax[j].set_xticklabels([f'x{s}' for s in SCALES])
    ax[j].set_title(f'{name} by scale and kernel (higher is better)')
    ax[j].set_ylabel(f'{name} {unit}'.strip())
    lo = min(res[(s, k)][j] for s in SCALES for k in KERNELS)
    hi = max(res[(s, k)][j] for s in SCALES for k in KERNELS)
    ax[j].set_ylim(lo - (hi - lo) * .15, hi + (hi - lo) * .15)
    ax[j].grid(axis='y', alpha=.3); ax[j].legend(fontsize=8)
plt.tight_layout(); plt.show()

best = {s: max(KERNELS, key=lambda k: res[(s, k)][0]) for s in SCALES}
print('배율별 PSNR 1위:', ', '.join(f'x{s} {best[s]}' for s in SCALES))

### 순위가 뒤집히는 지점

이 데이터에서는 **Lanczos 가 PSNR·SSIM 모두 1위**이고 Bicubic 이 근소하게 뒤따른다.
둘의 차이는 ×3 에서 0.1 dB 남짓으로 크지 않다.

주목할 것은 **Nearest 와 Bilinear 의 순위가 지표마다 뒤집힌다**는 점이다.
Bilinear 가 PSNR 은 더 높은데 SSIM 은 더 낮다. 뭉개서 평균에 가깝게 만들면 화소 오차
(PSNR)는 줄지만 구조(SSIM)는 잃기 때문이다.

**"PSNR 이 높다 = 보기 좋다" 가 아니다.** 이 어긋남은 딥러닝 SR 에서도 그대로 반복된다.

## 4. 확대해서 보기 — 계단, 흐림, 링잉

세 가지 결함이 커널마다 다르게 나타난다.

| 결함 | 어떻게 보이나 | 주로 어디서 |
|---|---|---|
| **계단 현상** | 비스듬한 선이 톱니처럼 각짐 | Nearest |
| **흐림** | 경계가 뭉개져 번짐 | Bilinear |
| **링잉** | 밝은 경계 옆에 어두운 띠(또는 그 반대)가 생김 | Bicubic, Lanczos |

가장 구조가 뚜렷한 사진에서 ×4 로 실험한 결과를 확대한다.

In [ ]:
def busiest(img, size):
    """경계가 가장 많은 구역 위치를 찾는다."""
    e = cv2.Canny(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), 50, 150)
    best, bs = (0, 0), -1
    for y in range(0, img.shape[0] - size, size // 2):
        for x in range(0, img.shape[1] - size, size // 2):
            v = e[y:y+size, x:x+size].mean()
            if v > bs: best, bs = (y, x), v
    return best

S, CROP = 4, 96
stem = max(HR, key=lambda n: cv2.Canny(cv2.cvtColor(HR[n], cv2.COLOR_RGB2GRAY), 50, 150).mean())
hr = HR[stem]
y, x = busiest(hr, CROP)
lr = downsample(hr, S)

panels = [('Original HR', hr)]
panels += [(k, upsample(lr, S, k)) for k in KERNELS]

fig, ax = plt.subplots(1, len(panels), figsize=(3.0 * len(panels), 3.4))
for a, (name, img) in zip(ax, panels):
    a.imshow(img[y:y+CROP, x:x+CROP], interpolation='nearest')
    t = name if name == 'Original HR' else f'{name}  {score(img, hr)[0]:.2f} dB'
    a.set_title(t, fontsize=9); a.set_xticks([]); a.set_yticks([])
fig.suptitle(f'x{S} restoration, zoomed {CROP}x{CROP} px  ({stem})', fontsize=10)
plt.tight_layout(); plt.show()

print('Nearest 의 톱니, Bilinear 의 번짐, Bicubic/Lanczos 의 경계 주변 띠를 비교해 보세요.')

## 5. 링잉을 수치로 보기

링잉은 **원본에 없던 값이 경계 주변에 생기는 것**이다. Bicubic 과 Lanczos 의 커널에는
음수 구간이 있어서, 밝은 곳과 어두운 곳이 만나면 실제보다 더 밝거나 더 어두운 값을
만들어낸다.

두 가지로 확인한다.

1. **경계 단면 그래프** — 밝기가 급변하는 한 줄을 뽑아 원본과 겹쳐 그린다
2. **오버슈트 비율** — 주변 3×3 원본 값의 범위를 벗어난 화소가 몇 %인지

In [ ]:
# 1) 가장 밝기 변화가 큰 가로줄을 하나 골라 단면을 그린다
g = cv2.cvtColor(hr, cv2.COLOR_RGB2GRAY).astype(np.float32)
row = int(np.argmax(np.abs(np.diff(g, axis=1)).max(axis=1)))
col = int(np.argmax(np.abs(np.diff(g[row]))))
x0, x1 = max(0, col - 12), min(g.shape[1], col + 13)

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(range(x0, x1), g[row, x0:x1], 'k-', lw=2.4, label='Original HR', zorder=5)
for k, c in zip(KERNELS, colors):
    up = cv2.cvtColor(upsample(lr, S, k), cv2.COLOR_RGB2GRAY).astype(np.float32)
    ax.plot(range(x0, x1), up[row, x0:x1], '-', color=c, lw=1.4, label=k)
ax.set_title(f'Brightness across a sharp edge (row {row})')
ax.set_xlabel('column'); ax.set_ylabel('brightness'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print('검은 선(원본) 위아래로 튀어나간 곡선이 링잉이다.')

In [ ]:
# 2) 보간에 쓰인 LR 화소들의 범위를 벗어난 출력 화소 비율
#    원본 HR 과 비교하면 링잉이 아니라 그냥 복원 오차를 재게 된다.
#    "입력에 없던 값을 만들어냈는가" 를 봐야 하므로 LR 쪽 범위를 기준으로 삼는다.
src = upsample(lr, S, 'Nearest')            # 각 출력 화소에 대응하는 LR 값
win = np.ones((2 * S + 1, 2 * S + 1), np.uint8)
lo = cv2.erode(src, win).astype(np.int16)
hi = cv2.dilate(src, win).astype(np.int16)

print(f'{"커널":10s} {"오버슈트":>10s} {"최대 초과폭":>12s} {"PSNR":>9s} {"SSIM":>9s}')
for k in KERNELS:
    up = upsample(lr, S, k).astype(np.int16)
    over = float(((up > hi + 2) | (up < lo - 2)).mean() * 100)
    amt = float(np.maximum(0, np.maximum(up - hi, lo - up)).max())
    p, q = res[(S, k)]
    print(f'{k:10s} {over:9.2f}% {amt:11.0f} DN {p:9.2f} {q:9.4f}')

print('\nNearest 와 Bilinear 는 정확히 0 이다. 주변 값을 섞기만 하므로 범위를 못 벗어난다.')
print('Bicubic 과 Lanczos 는 커널에 음수 구간이 있어 입력에 없던 값을 만들어낸다. 이것이 링잉이다.')

## 6. Baseline 확보

여기까지는 **HR 을 내가 직접 줄여서** LR 을 만들었다. 그런데 딥러닝 SR 노트북이 쓰는
LR(`g_LR`)은 다른 방식으로 만들어진 것이라, 두 수치를 그냥 비교하면 안 된다.

그래서 **데이터셋의 실제 `g_LR` 로도 똑같이 재서** 나란히 놓는다. 이 쪽이 딥러닝 SR 과
직접 비교할 수 있는 값이다.

In [ ]:
# 데이터셋의 실제 LR 로도 재본다 (10장 x 40KB 남짓)
os.makedirs('lr', exist_ok=True)
real = {}
for n in names:
    fn = f'{n[:-4]}x3.png'
    if not os.path.exists(f'lr/{fn}'):
        urllib.request.urlretrieve(f'{BASE}/dataset/validation/LR_bicubic/X3/{fn}', f'lr/{fn}')
    real[n[:-4]] = imageio.imread(f'lr/{fn}')

res_real = {}
for k in KERNELS:
    ps, ss = [], []
    for stem, hr_ in HR.items():
        up = upsample(real[stem], 3, k)
        p, q = score(up, hr_)
        ps.append(p); ss.append(q)
    res_real[k] = (float(np.mean(ps)), float(np.mean(ss)))

print(f'{"커널":10s} {"내가 줄인 LR":>22s} {"데이터셋 g_LR":>22s}')
for k in KERNELS:
    a, b = res[(3, k)], res_real[k]
    print(f'{k:10s}  PSNR {a[0]:6.2f}  SSIM {a[1]:.4f}   PSNR {b[0]:6.2f}  SSIM {b[1]:.4f}')

gap = res[(3, 'Bicubic')][0] - res_real['Bicubic'][0]
print(f'\n같은 x3 인데 {gap:.2f} dB 차이가 난다.')
print('LR 을 어떻게 만드느냐가 문제의 난이도를 지배한다는 뜻이다.')
print('논문마다 SR 점수가 크게 다른 이유도 대부분 여기에 있다.')

In [ ]:
baseline = {f'x{s}': {k: {'psnr': round(res[(s, k)][0], 3),
                          'ssim': round(res[(s, k)][1], 4)} for k in KERNELS}
            for s in SCALES}
baseline['x3_dataset_gLR'] = {k: {'psnr': round(res_real[k][0], 3),
                                  'ssim': round(res_real[k][1], 4)} for k in KERNELS}
with open('baseline_interpolation.json', 'w') as f:
    json.dump({'n_images': len(HR), 'shave': SHAVE,
               'downsample': 'INTER_AREA (x3_dataset_gLR 만 데이터셋 제공 LR)',
               'result': baseline}, f, indent=1)

print('=== 보간법 baseline (검증 10장 평균) ===')
for s in SCALES:
    mark = '   (HR 을 INTER_AREA 로 줄인 LR)' if s == 3 else ''
    print(f'\nx{s}{mark}')
    for k in KERNELS:
        p, q = res[(s, k)]
        print(f'   {k:10s} PSNR {p:6.2f} dB   SSIM {q:.4f}')

print('\nx3, 데이터셋 g_LR   <- 딥러닝 SR 과 직접 비교할 값')
for k in KERNELS:
    p, q = res_real[k]
    print(f'   {k:10s} PSNR {p:6.2f} dB   SSIM {q:.4f}')

rb = res_real['Bicubic']
print(f'\n기억할 숫자: x3 Bicubic(g_LR) = PSNR {rb[0]:.2f} dB / SSIM {rb[1]:.4f}')
print('baseline_interpolation.json 으로 저장했다.')

---

## 정리

- **배율이 커지면 점수가 떨어진다.** ×4 는 화소의 15/16 을 버리는 셈이라 되살릴 수가 없다
- **커널마다 실패하는 방식이 다르다.** Nearest 는 각지고, Bilinear 는 뭉개지고,
  Bicubic·Lanczos 는 경계에 없던 띠를 만든다
- **PSNR 1위와 보기 좋은 것이 다를 수 있다.** 흐릿하면 오차는 작지만 디테일은 없다
- 보간법은 **있는 정보를 매끄럽게 이어붙일 뿐, 없는 디테일을 만들지 못한다.**
  딥러닝 SR 은 "이런 흐린 패턴은 보통 이런 선명한 모양이었다"를 학습해서 그 지점을 넘는다

- **LR 을 만드는 방식이 점수를 지배한다.** 같은 ×3 인데 4.7 dB 나 차이가 났다.
  SR 결과를 비교할 때는 항상 같은 방식으로 만든 LR 인지부터 확인해야 한다

다음: [`01_edsr_x3.ipynb`](https://colab.research.google.com/github/BWMIN-Hub/SR_practice/blob/main/notebooks/01_edsr_x3.ipynb)
에서 딥러닝 SR 이 위 **x3 / 데이터셋 g_LR** 기준선을 얼마나 넘는지 확인한다.